This notebook initializes a Document Processing Pipeline using LangChain and Groq. It automatically installs dependencies, loads PDF documents from a local /data directory, and performs text chunking. Finally, it generates vector embeddings using a HuggingFace model and stores them in a ChromaDB vector store for efficient semantic retrieval.

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Install necessary libraries for LLM orchestration, vector storage, and PDF parsing
!{sys.executable} -m pip install -U langchain-community langchain-core langchain-groq langchain-huggingface chromadb pypdf python-dotenv

# Import components for LLM connectivity, embeddings, and document handling
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Load environment variables (API keys) from a .env file
load_dotenv()

# Set the Groq API key for authentication with the inference engine
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Data Directory Management: Ensure the 'data' folder exists to store source PDFs
if not os.path.exists("./data"):
    os.makedirs("./data")
    print("📂 Created 'data' folder. Please add your PDFs!")
else:
    docs = []
    # Loop through the data folder and load every PDF found
    for file in os.listdir("./data"):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join("./data", file))
            docs.extend(loader.load())
    
    if docs:
        # Text Splitting: Break long documents into chunks to fit LLM context windows
        # chunk_overlap ensures the AI maintains context between adjacent chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        splits = text_splitter.split_documents(docs)
        
        # Embedding Initialization: Use a lightweight local model to turn text into math vectors
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        
        # Vector Database: Create and populate ChromaDB with the document chunks
        vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
        print(f"✅ Success! Created {len(splits)} chunks using GROQ_API_KEY from .env.")
    else:
        print("⚠️ Folder is empty. Please add PDF files to './data'.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Success! Created 3896 chunks using GROQ_API_KEY from .env.


This cell defines the RAG (Retrieval-Augmented Generation) Logic by creating a custom chain. It uses a vector retriever to pull relevant snippets, formats them into a structured prompt, and passes the context to the llama-3.3-70b-versatile model. The logic is specifically tuned for comparative data analysis and ranking tasks.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize the LLM: Using Llama 3.3 70B via Groq for high speed and reasoning
# temperature=0 ensures the model stays factual and doesn't "hallucinate" creative data
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

def pdf_chatbot(query):
    try:
        # Retriever Config: Fetch the top 3 most relevant document chunks for any query
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        
        # Prompt Engineering: Instructions to force the model into a "Data Analyst" mindset
        # This specific prompt focuses on extracting numerical values and ranking them
        prompt = ChatPromptTemplate.from_template("""
        You are a Senior Economic Data Analyst. Use the provided context to answer the user's request.
        
        TASK:
        1. If the user asks for a "leader" or "highest/lowest," extract ALL countries and their percentages first.
        2. Rank them numerically from highest to lowest.
        3. Identify the true leader (highest value) and compare it specifically to any other country mentioned in the question.
        4. Keep the response to 4 concise, professional lines.

        Context: {context}
        Question: {question}
        
        Answer:""")

        # Helper Function: Join multiple document chunks into a single string for the prompt
        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        # LCEL Chain Construction:
        # - Pass the question through to the retriever to get context
        # - Pipe ( | ) the formatted context and question into the prompt
        # - Pipe the prompt into the LLM
        # - Pipe the LLM output into a string parser for clean text
        rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )

        # Execute the chain and return the AI's response
        return rag_chain.invoke(query)

    except Exception as e:
        # Error handling in case of API issues or empty vector stores
        return f"⚠️ Error in Analysis: {str(e)}"

This cell builds the Interactive Chat Interface. It utilizes ipywidgets to create a responsive front-end with a dedicated input bar, send/clear buttons, and a stylized message history. The UI includes JavaScript-based auto-scrolling and HTML/CSS styling to provide a polished, "ChatGPT-like" experience for interacting with your processed PDFs.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Javascript

# --- Global History ---
# Stores the conversation thread so it persists during the session
chat_history = []

# UI Component Setup
# Text input box where the user types their questions
text_input = widgets.Text(
    placeholder='Ask your PDFs...', 
    layout={'width': '82%'}
)

# Success-styled button with a paper-plane icon for sending queries
send_btn = widgets.Button(
    description="", 
    button_style='success', 
    icon='paper-plane',
    layout={'width': '45px'}
)

# Danger-styled button for resetting the conversation
clear_btn = widgets.Button(
    description="", 
    button_style='danger', 
    icon='trash',
    layout={'width': '45px'},
    tooltip="Clear Chat"
)

# Output Area (The Chat Window)
# HTML widget used to render the stylized chat bubbles
output_html = widgets.HTML(
    value="<div style='color:#999; text-align:center; padding-top:100px;'>Ready to chat!</div>"
)

# Container for the HTML widget with specific styling for height, borders, and scrolling
output_container = widgets.Box(
    [output_html], 
    layout={
        'border': '1px solid #ddd', 
        'padding': '15px', 
        'height': '400px', 
        'overflow_y': 'auto', 
        'width': '100%',
        'background-color': '#ffffff',
        'border_radius': '8px'
    }
)

def update_display():
    """Renders the current chat_history list into HTML bubbles."""
    if not chat_history:
        output_html.value = "<div style='color:#999; text-align:center; padding-top:100px;'>Chat cleared.</div>"
        return

    full_chat = "<div style='width: 100%;'>"
    for index, chat in enumerate(chat_history):
        # User Bubble: Blue background, right-aligned
        full_chat += f"<div style='margin-bottom: 12px; display: flex; justify-content: flex-end;'>"
        full_chat += f"<div style='background-color: #007bff; color: white; padding: 10px 14px; border-radius: 15px 15px 0 15px; font-size: 13.5px; max-width: 85%; font-family: sans-serif; box-shadow: 1px 1px 3px rgba(0,0,0,0.1);'>"
        full_chat += f"<b>You:</b> {chat['user']}</div></div>"
        
        # Assistant Bubble: Light gray background, left-aligned
        full_chat += f"<div id='msg-{index}' style='margin-bottom: 25px; display: flex; justify-content: flex-start;'>"
        full_chat += f"<div style='background-color: #f8f9fa; border: 1px solid #e9ecef; padding: 10px 14px; border-radius: 15px 15px 15px 0; font-size: 13.5px; max-width: 90%; font-family: sans-serif; box-shadow: 1px 1px 3px rgba(0,0,0,0.05);'>"
        full_chat += f"<b style='color: #28a745;'>Assistant:</b><br>{chat['bot'].replace('\n', '<br>')}</div></div>"
    full_chat += "</div>"
    
    output_html.value = full_chat
    
    # Auto-scroll: Injects JS to force the scrollbar to the bottom after a message
    display(Javascript("""
        var boxes = document.querySelectorAll('.widget-box');
        boxes.forEach(function(box) {
            box.scrollTop = box.scrollHeight;
        });
    """))

def process_query(sender):
    """Triggered when user clicks send or presses Enter."""
    user_query = text_input.value
    if user_query.strip():
        # Change icon to spinner to show processing state
        send_btn.icon = "spinner"
        
        # Connects the UI to the RAG logic defined in the previous cell
        result = pdf_chatbot(user_query) 
        
        # Update history and refresh the UI
        chat_history.append({"user": user_query, "bot": result})
        update_display()
        
        # Reset input and icon
        text_input.value = ""
        send_btn.icon = "paper-plane"

def clear_chat(b):
    """Wipes the session history."""
    global chat_history
    chat_history = []
    update_display()

# Event Binding: Linking UI actions to Python functions
send_btn.on_click(process_query)
text_input.on_submit(process_query)
clear_btn.on_click(clear_chat)

# --- Final Layout ---
# Arrange the widgets into a vertical (VBox) and horizontal (HBox) layout
input_bar = widgets.HBox([text_input, send_btn, clear_btn], layout={'margin': '0 0 10px 0'})
final_ui = widgets.VBox([input_bar, output_container], layout={'width': '750px', 'margin': '0 auto'})

print("✅ UI Ready for Demo: Input on top, width optimized.")
display(final_ui)

✅ UI Ready for Demo: Input on top, width optimized.


C:\Users\qasim\AppData\Local\Temp\ipykernel_13408\1037449652.py:94: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(process_query)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

This notebook successfully integrates LangChain, Groq, and ChromaDB to create an end-to-end PDF analysis tool. By combining local vector embeddings with the Llama-3.3-70b model, the system delivers high-speed, context-aware answers through a custom-built interactive UI. This setup provides a scalable foundation for specialized data extraction, ranking, and document intelligence tasks directly within a Jupyter environment.